# Thesis Results Summary — Multimodal Prediction of Perceived Effectiveness

Curated, thesis-ready subset of results from `prediction_models_v2.ipynb`. This notebook does not
recompute any models — it loads the already-saved result tables and presents only the findings that
are (a) at the correct primary unit of analysis (group-task observation, per `docs/thesis_research_questions.md`),
(b) FDR-corrected across the relevant family of tests, and (c) not confounded by a plain task main effect.
Null / exploratory results are reported honestly rather than omitted.

## Methodology (as run in `prediction_models_v2.ipynb`)

- **Unit of analysis (primary):** group-task observation — 10 groups x up to 3 tasks (~18-30 rows
  depending on target/task subset), matching the thesis's stated primary unit.
- **Cross-validation:** `LeaveOneGroupOut` (LOGO) and grouped `GroupKFold` (L2GO), grouped by `group_id`
  so no group ever appears in both train and test.
- **Model:** Ridge regression (`RidgeCV`) on standardized features.
- **Significance:** permutation testing (100 permutations of the target within CV), then
  Benjamini-Hochberg FDR correction applied within each CV scheme across the full 95-model x
  2-CV-scheme ablation grid (`Section 14`, 190 tests total).
- **Secondary, exploratory layer:** participant-level Mixed-effects (MixedLM) likelihood-ratio tests
  (`Sections 16b/16c/17`) — these use participant-level rows and are explicitly outside the thesis's
  primary group-task-level design; treated here as exploratory/discussion material, not confirmatory.
- **High-resolution follow-up (`Section 18`):** for the two headline findings below, the permutation count was raised to 2000 (from 100) and a group-level bootstrap 95% CI on R2 was added (resample `group_id`s with replacement, rebuild LOGO/L2GO folds, refit) — this gives a much finer p-value and a sense of estimate stability that the original 100-permutation ablation could not provide.


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.width', 200)

REPO_ROOT    = Path.cwd() if (Path.cwd() / 'icmi_paper').exists() else Path.cwd().parent
ANALYSIS_RES = REPO_ROOT / 'analysis' / 'results'

model_comp = pd.read_csv(ANALYSIS_RES / 'prediction_model_comparison_v20260905.tsv', sep='\t')
print(f'{len(model_comp)} rows loaded '
      f'({model_comp["Model"].nunique()} feature sets x {model_comp["Target"].nunique()} targets '
      f'x {model_comp["CV"].nunique()} CV schemes)')

def show(target, models=None, cv=None):
    sub = model_comp[model_comp['Target'] == target]
    if models is not None:
        sub = sub[sub['Model'].isin(models)]
    if cv is not None:
        sub = sub[sub['CV'] == cv]
    return sub.sort_values(['CV', 'q'])[['CV', 'Model', 'R2', 'n', 'p', 'q']]

190 rows loaded (19 feature sets x 5 targets x 2 CV schemes)


## Finding 1 (primary, H2-aligned): conversation-structure features predict `team_coordination`

`team_coordination`'s task-mean-only baseline is **not** significant (LOGO p=0.327, L2GO p=0.644) — so
there is no task-level confound to control for here. Turn-taking / conversation-structure feature sets
are FDR-significant and the `Turn + Lex + DA` combination replicates under **both** CV schemes:

In [2]:
show('team_coordination', models=[
    'Baseline (task-mean only)', 'Turn-taking only', 'Turn + Lex + DA', 'Expanded raw',
])

,CV,Model,R2,n,p,q
155,L2GO,Expanded raw,0.176,17,0.010,0.048
180,L2GO,Turn + Lex + DA,0.237,18,0.010,0.048
120,L2GO,Turn-taking only,0.144,18,0.030,0.073
95,L2GO,Baseline (task-mean only),-0.177,20,0.644,0.765
25,LOGO,Turn-taking only,0.296,18,0.010,0.040
85,LOGO,Turn + Lex + DA,0.484,18,0.010,0.040
60,LOGO,Expanded raw,-0.010,17,0.059,0.104
0,LOGO,Baseline (task-mean only),-0.130,20,0.327,0.420


In [3]:
highres = pd.read_csv(ANALYSIS_RES / 'headline_findings_highres_v20260908.tsv', sep='\t')
highres[highres['Target'] == 'team_coordination']

,Model,Target,CV,R2,n,p_2000perm,boot_ci_2.5%,boot_ci_97.5%,n_boot_valid
2,Turn-taking only,team_coordination,LOGO,0.296,18,0.007,-0.462,0.959,1000
3,Turn-taking only,team_coordination,L2GO,0.144,18,0.017,-0.642,0.938,1000


**Interpretation:** group-level conversation-structure/participation measures (speaking time,
overlap, backchannel counts) add real predictive value for `team_coordination` beyond chance and beyond
task alone. This directly supports the thesis's H2 ("conversation structure and participation measures
are expected to be most relevant to group-functioning outcomes") at the correct group-task unit of
analysis.

**Caveat (high-resolution follow-up):** with 2000 permutations, p tightens to 0.007 (LOGO) / 0.017
(L2GO) — still nominally significant. However, the group-bootstrap 95% CI on R2 is wide and **straddles
zero** in both CV schemes (LOGO [-0.462, 0.959], L2GO [-0.642, 0.938]), reflecting the small sample
(n=18 group-task rows for a 13-feature model). Report this as **suggestive, not confirmatory** —
consistent with an earlier LOGO/L2GO R2 discrepancy (0.296 vs 0.144) for the same feature set.


## Finding 2 (H2-aligned): turn-taking features predict `voice_inclusion` beyond the task baseline

Unlike `team_coordination`, `voice_inclusion`'s baseline **is** FDR-significant (LOGO R2=0.357 p=0.01,
L2GO R2=0.362 p=0.01) — the target has a real task main effect, so only feature sets that **exceed**
the baseline R2 represent added behavioral value. `Overlap+Backchannel` (turn-taking) clears that bar
under both CV schemes:

In [4]:
show('voice_inclusion', models=[
    'Baseline (task-mean only)', 'Overlap+Backchannel (rescue)', 'Turn-taking only', 'Audio only',
])

,CV,Model,R2,n,p,q
96,L2GO,Baseline (task-mean only),0.362,30,0.01,0.048
116,L2GO,Audio only,0.493,27,0.01,0.048
121,L2GO,Turn-taking only,0.280,28,0.01,0.048
186,L2GO,Overlap+Backchannel (rescue),0.483,28,0.01,0.048
1,LOGO,Baseline (task-mean only),0.357,30,0.01,0.040
21,LOGO,Audio only,0.275,27,0.01,0.040
91,LOGO,Overlap+Backchannel (rescue),0.459,28,0.01,0.040
26,LOGO,Turn-taking only,0.299,28,0.02,0.056


In [5]:
highres[highres['Target'] == 'voice_inclusion']

,Model,Target,CV,R2,n,p_2000perm,boot_ci_2.5%,boot_ci_97.5%,n_boot_valid
0,Overlap+Backchannel (rescue),voice_inclusion,LOGO,0.459,28,0.001,0.201,0.769,1000
1,Overlap+Backchannel (rescue),voice_inclusion,L2GO,0.483,28,0.001,0.095,0.769,1000


**Interpretation:** `tr_ovl_count` (count of overlapping-speech events) and `tr_backchannel_count`
(count of short supportive interjections such as "mm-hmm", per `docs/feature_catalog.md` §4) predict
`voice_inclusion` above and beyond the task-only baseline — a second, independent piece of H2 support.

**High-resolution follow-up:** with 2000 permutations, p tightens to **0.001** under both CV schemes
(from a floor of ≤0.01 with the original 100 permutations). The group-bootstrap 95% CI on R2 **excludes
zero** in both schemes (LOGO [0.201, 0.769], L2GO [0.095, 0.769]) — this is the most robust finding in
the whole analysis and the one to lead with in the thesis. By conventional benchmarks (Cohen, 1988;
R2 ≈ 0.02/0.13/0.26 = small/medium/large), R2 ≈ 0.46-0.48 is a **large** effect.

**Construct-validity / collinearity check (`Section 18b`):** `tr_ovl_count` and `tr_backchannel_count`
correlate only weakly with each other (r=0.20) and with `tr_silence_duration_s` (r≈-0.17 to 0.18) —
so neither is simply proxying "how much the group talked overall", and the two-feature result is not a
multicollinearity artifact. Each feature *individually* already reaches p=0.001 with R2≈0.42-0.47 (2000
permutations), nearly matching the two-feature model's R2=0.46-0.48 — both are independently, robustly
associated with `voice_inclusion`, and the combined model is a convergent, not fragile, result.


## Caveat: `mental_demand` and `satisfaction` are largely explained by task alone

Both targets' baselines are FDR-significant, and none of the "significant" behavioral feature sets
actually **beat** the baseline R2 for `mental_demand` — meaning the apparent significance mostly
reflects that `mental_demand` differs systematically by task, not that behavioral features predict it
beyond task. `satisfaction` has one feature set that beats baseline (`Semantic + completion`,
L2GO R2=0.518 vs baseline 0.356), but it is built on only 14 observations — likely underpowered/fragile,
not a claim to lean on.

In [6]:
for target in ['mental_demand', 'satisfaction']:
    print(f'--- {target}: baseline ---')
    print(show(target, models=['Baseline (task-mean only)']).to_string(index=False))
    print(f'--- {target}: top 3 non-baseline by R2 ---')
    best = model_comp[(model_comp['Target'] == target) & (model_comp['Model'] != 'Baseline (task-mean only)')]
    print(best.sort_values('R2', ascending=False).head(3)[['CV', 'Model', 'R2', 'n', 'p', 'q']].to_string(index=False))
    print()

--- mental_demand: baseline ---
  CV                     Model    R2  n    p     q
L2GO Baseline (task-mean only) 0.422 30 0.01 0.048
LOGO Baseline (task-mean only) 0.410 30 0.01 0.040
--- mental_demand: top 3 non-baseline by R2 ---
  CV                         Model    R2  n    p     q
LOGO                    Audio only 0.387 27 0.01 0.040
L2GO HMM states new K5 (v20260905) 0.336 28 0.01 0.048
LOGO HMM states new K5 (v20260905) 0.323 28 0.01 0.040

--- satisfaction: baseline ---
  CV                     Model    R2  n    p     q
L2GO Baseline (task-mean only) 0.356 20 0.01 0.048
LOGO Baseline (task-mean only) 0.368 20 0.01 0.040
--- satisfaction: top 3 non-baseline by R2 ---
  CV                 Model    R2  n    p     q
L2GO Semantic + completion 0.518 14 0.01 0.048
LOGO Semantic + completion 0.454 14 0.02 0.056
LOGO           Physio only 0.343 19 0.02 0.056



## `coord+coop` (composite cooperative/coordination target): weak but present

Baseline is not significant here either, so the FDR-significant hits are clean, but effect sizes are
small (R2 ~0.10-0.15, n=28) — worth reporting as a minor supporting result, not a headline one.

In [7]:
show('coord+coop', models=[
    'Baseline (task-mean only)', 'States + Lex + DA', 'Turn + Lex + DA', 'Overlap+Backchannel (rescue)',
])

,CV,Model,R2,n,p,q
188,L2GO,Overlap+Backchannel (rescue),0.154,28,0.010,0.048
183,L2GO,Turn + Lex + DA,0.086,28,0.020,0.058
98,L2GO,Baseline (task-mean only),-0.119,30,0.634,0.765
178,L2GO,States + Lex + DA,-0.377,28,0.624,0.765
83,LOGO,States + Lex + DA,0.099,28,0.010,0.040
88,LOGO,Turn + Lex + DA,0.098,28,0.010,0.040
93,LOGO,Overlap+Backchannel (rescue),0.046,28,0.040,0.090
3,LOGO,Baseline (task-mean only),-0.121,30,0.673,0.761


## Follow-up (`prediction_models_v2.ipynb` Section 19): does feature reduction rescue `team_coordination`/`coord+coop`?

The 2-feature reduction that rescued `voice_inclusion` (Finding 2 above) was tested on
`team_coordination` and `coord+coop` too, since both have unstable 13-feature results. A single-feature
scan across all 13 `Turn-taking only` features found only overlap-timing measures with nominal p<0.10,
and inconsistently across CV schemes:

In [8]:
scan = pd.read_csv(ANALYSIS_RES / 'turn_taking_single_feature_scan_v20260908.tsv', sep='\t')
print(scan[scan['p'] < 0.10].sort_values(['Target', 'CV'])[['Target', 'CV', 'Feature', 'R2', 'n', 'p', 'q']].to_string(index=False))

               Target   CV                   Feature     R2  n     p        q
coord+coop (centered) L2GO              tr_ovl_count  0.025 28 0.050 0.325000
coord+coop (centered) L2GO             tr_ovl_time_s  0.023 28 0.032 0.325000
coord+coop (centered) LOGO              tr_ovl_count -0.063 28 0.092 0.598000
coord+coop (centered) LOGO tr_ovl_competitive_timing -0.021 28 0.068 0.598000
           team_coord L2GO             tr_ovl_time_s  0.156 18 0.008 0.065000
           team_coord L2GO tr_ovl_competitive_timing  0.272 18 0.010 0.065000
           team_coord L2GO              tr_ovl_count  0.016 18 0.038 0.164667
           team_coord LOGO tr_ovl_competitive_timing  0.224 18 0.010 0.130000
           team_coord LOGO              tr_ovl_count -0.020 18 0.068 0.294667
           team_coord LOGO             tr_ovl_time_s -0.056 18 0.052 0.294667


In [9]:
rescue = pd.read_csv(ANALYSIS_RES / 'team_coord_coopcoop_rescue_v20260908.tsv', sep='\t')
print(rescue.to_string(index=False))

           Target                                               Features   CV    R2  n  p_2000perm  boot_ci_2.5%  boot_ci_97.5%
team_coordination tr_ovl_count, tr_ovl_time_s, tr_ovl_competitive_timing LOGO 0.233 18       0.015        -0.741          0.686
team_coordination tr_ovl_count, tr_ovl_time_s, tr_ovl_competitive_timing L2GO 0.242 18       0.012        -0.920          0.681
       coord+coop tr_ovl_count, tr_ovl_time_s, tr_ovl_competitive_timing LOGO 0.066 28       0.031        -0.279          0.558
       coord+coop tr_ovl_count, tr_ovl_time_s, tr_ovl_competitive_timing L2GO 0.073 28       0.022        -0.414          0.547

**Interpretation:** re-testing the best-supported 3-feature subset
(`tr_ovl_count`, `tr_ovl_time_s`, `tr_ovl_competitive_timing`) with the same 2000-permutation +
bootstrap-CI treatment as Finding 2 did **not** rescue either target. `team_coordination` reaches
R2=0.23-0.24 (p=0.01-0.015), but its bootstrap CI is still wide and straddles zero (LOGO [-0.741, 0.686],
L2GO [-0.920, 0.681]) — actually *wider* than the original 13-feature model's CI. `coord+coop` shows the
same pattern with a weaker R2 (~0.07). **Conclusion:** unlike `voice_inclusion`, where noise-feature
dilution was the problem, `team_coordination`/`coord+coop`'s instability reflects a genuine small
effect size at n=18-28 group-task rows, not a feature-selection problem. Feature reduction does not
rescue these targets — they remain **suggestive, not confirmatory**, and no further rescue attempts are
planned (H5/H6 temporal-organization/state-stability tests were also considered and deprioritized for
the same reason: low expected payoff relative to the added analysis risk on an already-thin sample).

## H3: do parsimonious raw-feature models generalize better than a high-dimensional raw baseline?

Per `docs/thesis_research_questions.md` H3, small theory-led models are expected to generalize more
reliably than a broad, high-dimensional raw-feature model. Comparing `Turn-taking only` (13 features)
against `Expanded raw` (Physio+ET+Audio+Turn-taking, ~20+ features) across all 5 primary targets:

In [10]:
for target in model_comp['Target'].unique():
    sub = model_comp[(model_comp['Target'] == target) &
                      (model_comp['Model'].isin(['Turn-taking only', 'Expanded raw']))]
    print(f'--- {target} ---')
    print(sub.sort_values(['CV', 'Model'])[['CV', 'Model', 'R2', 'n', 'p', 'q']].to_string(index=False))
    print()

--- team_coordination ---
  CV            Model     R2  n     p     q
L2GO     Expanded raw  0.176 17 0.010 0.048
L2GO Turn-taking only  0.144 18 0.030 0.073
LOGO     Expanded raw -0.010 17 0.059 0.104
LOGO Turn-taking only  0.296 18 0.010 0.040

--- voice_inclusion ---
  CV            Model     R2  n     p     q
L2GO     Expanded raw -0.730 27 0.792 0.865
L2GO Turn-taking only  0.280 28 0.010 0.048
LOGO     Expanded raw  0.402 27 0.010 0.040
LOGO Turn-taking only  0.299 28 0.020 0.056

--- mental_demand ---
  CV            Model     R2  n     p     q
L2GO     Expanded raw -0.058 27 0.168 0.275
L2GO Turn-taking only -0.069 28 0.149 0.253
LOGO     Expanded raw -0.017 27 0.059 0.104
LOGO Turn-taking only  0.152 28 0.010 0.040

--- coord+coop ---
  CV            Model     R2  n     p     q
L2GO     Expanded raw -0.300 27 0.653 0.766
L2GO Turn-taking only  0.016 28 0.099 0.188
LOGO     Expanded raw -2.794 27 0.950 0.950
LOGO Turn-taking only  0.019 28 0.050 0.095

--- satisfaction ---
  CV

**Interpretation:** supports H3, but via instability rather than a clean win. `Expanded raw` never
beats `Turn-taking only` by a robust margin under both CV schemes simultaneously, and it collapses
badly on several targets (e.g. `coord+coop` LOGO R2=-2.794; `voice_inclusion` L2GO R2=-0.730) — classic
overfitting given only ~18-28 rows for a wider feature set. `Turn-taking only` never craters this badly,
even when its own R2 is not significant. **Report as:** the high-dimensional raw baseline is not more
reliable than the parsimonious model, and is frequently much less reliable — consistent with the
thesis's stated concern about model complexity given the small number of independent groups.

## H4: does the HMM state representation match or exceed raw/PCA baselines?

Per H4, comparing `HMM states new K5 (v20260905)` against the task-mean baseline and against
`Turn-taking only` across all 5 targets:

In [11]:
for target in model_comp['Target'].unique():
    sub = model_comp[(model_comp['Target'] == target) &
                      (model_comp['Model'].isin(['Baseline (task-mean only)', 'HMM states new K5 (v20260905)', 'Turn-taking only']))]
    print(f'--- {target} ---')
    print(sub.sort_values(['CV', 'Model'])[['CV', 'Model', 'R2', 'n', 'p', 'q']].to_string(index=False))
    print()

--- team_coordination ---
  CV                         Model     R2  n     p     q
L2GO     Baseline (task-mean only) -0.177 20 0.644 0.765
L2GO HMM states new K5 (v20260905) -0.248 18 0.446 0.614
L2GO              Turn-taking only  0.144 18 0.030 0.073
LOGO     Baseline (task-mean only) -0.130 20 0.327 0.420
LOGO HMM states new K5 (v20260905) -0.173 18 0.238 0.328
LOGO              Turn-taking only  0.296 18 0.010 0.040

--- voice_inclusion ---
  CV                         Model     R2  n     p     q
L2GO     Baseline (task-mean only)  0.362 30 0.010 0.048
L2GO HMM states new K5 (v20260905) -0.199 28 0.733 0.829
L2GO              Turn-taking only  0.280 28 0.010 0.048
LOGO     Baseline (task-mean only)  0.357 30 0.010 0.040
LOGO HMM states new K5 (v20260905)  0.076 28 0.040 0.090
LOGO              Turn-taking only  0.299 28 0.020 0.056

--- mental_demand ---
  CV                         Model     R2  n     p     q
L2GO     Baseline (task-mean only)  0.422 30 0.010 0.048
L2GO HMM state

**Interpretation:** null/mixed result for H4 (which the thesis explicitly treats as informative,
not disqualifying). The HMM state representation does not exceed the raw-feature baselines on the two
headline targets (`team_coordination`, `voice_inclusion`) — it underperforms `Turn-taking only` and, for
`team_coordination`, underperforms even the null baseline. Its one genuinely competitive result is
`mental_demand` (HMM R2=0.323-0.336, close to but still below the baseline's 0.410-0.422) — but recall
from the caveat above that `mental_demand`'s baseline itself reflects a task main effect, not behavioral
signal, so this is not strong evidence that the HMM states add unique value there either. **Report as:**
the constrained HMM does not provide a predictive advantage over simpler representations in this
dataset — a valid, thesis-consistent null result for H4, not a modeling failure.

## Latent-state representation comparison: HMM vs. K-Means vs. Continuous PCA (RQ3/H4)

All three representations below are derived from the **identical** 10-feature, high-coverage window-level
feature set (`docs/thesis_research_questions.md` §6 primary HMM candidate families), so any difference in
performance is attributable to the representation itself, not the underlying signal:

- **HMM states (k=5):** discrete, temporal/Markov-constrained regimes (the thesis's primary latent-state model).
- **K-Means clusters (k=5):** discrete, non-temporal — same number of clusters, no sequence structure.
- **Continuous PCA (no discretization):** the raw principal components themselves, no clustering step at all.

Comparing all three against the task-mean baseline isolates two questions at once: does *discretizing
into regimes at all* help, and does *adding temporal/Markov structure* help beyond plain clustering?

### Qualitative characterization of the states (k=5)

Before testing whether the states carry predictive value, it is worth establishing what they
actually represent. All five states are populated in 9-10/10 groups (none is a single-group
artifact), fit on 616 T1-T3 windows via Baum-Welch/Viterbi on the same 10-feature input
(`docs/hmm_state_interpretation.md` §6a has the full methodology and per-feature z-scores):

| State | n windows (%) | Dominant task | Profile / interpretation |
|---|---|---|---|
| **S0** | 74 (12%) | T2 82%, T1 18% | Elevated EDA (+0.76), temp (+0.73), pupil (+0.93) but vocally withdrawn (backchannel -0.41, overlap -0.78, competitive overlap -0.81, laughter -0.79) — **tense quiet / internal deliberation**: physiologically aroused but silent, consistent with held-back tension or silent calculation during negotiation. |
| **S1** | 199 (32%, largest) | T3 61% | Highest backchannel (+1.29), laughter (+1.24), competitive overlap (+1.09), overlap (+0.97) of any state, temp lowest (-1.53) — **animated/collaborative exchange**: a brainstorming register with lots of overlapping, playful talk, dominant during idea generation. |
| **S2** | 117 (19%) | T2 82% | Highest HR (+0.86) of any state, co-occurring with the highest overlap (+1.19) and competitive overlap (+1.08) — **heated/contested exchange**: the classic physiological-plus-vocal signature of a contested negotiation moment. |
| **S3** | 114 (19%) | T1 54% | Lowest HR (-1.57), EDA (-1.72), pupil (-1.43) of any state; conversational features near-neutral — **low-arousal baseline**: the physiologically calmest state, dominant during the earliest task (hidden-profile decision). |
| **S4** | 112 (18%) | spread: T1 29%, T2 43%, T3 28% | Silence (-1.76), backchannel (-1.36), active speakers (-1.53) all strongly suppressed; raw overlap/competitive overlap/laughter all ≈0 — **non-speech / task-transition**: essentially the "no dialogue captured" state (instructions, individual work, inter-phase gaps), elevated physiological arousal but no discourse. |

**Transition dynamics:** all five states are *sticky* (diagonal self-transition probability
0.51-0.91), i.e. sustained multi-window regimes rather than rapid flicker. S1 is the most stable
(0.91 self-transition, almost never exited). S0 and S2 — the two T2-dominant states — interconvert
with each other more than with any other state (S0→S2 0.35, S2→S0 0.16), suggesting groups
oscillate between quiet tension (S0) and active contest (S2) within the negotiation task
specifically. S3 and S4 are each largely self-contained (0.83 and 0.86 self-transition) with
little cross-traffic to the other states.

**Caveat:** these labels are post-fit, human-assigned interpretations of feature means, not
ground truth, and quantitative self-report validation (Spearman correlation of state proportions
against self-report items, per `docs/hmm_state_interpretation.md` §7-§8) has **not yet been
re-run** on this specific 2026-09-05 refit. Treat this as a qualitative characterization only,
distinct from and prior to the quantitative predictive comparison below and the H6 stability
check further down, which are the tests that actually bear on RQ3/H4/H6.

In [12]:
latent_models = ['Baseline (task-mean only)', 'HMM states new K5 (v20260905)',
                 'K-Means clusters (no temporal, v20260905)', 'Continuous PCA (no discretization, v20260905)']
for target in ['team_coordination', 'voice_inclusion', 'mental_demand', 'coord+coop', 'satisfaction']:
    print(f'--- {target} ---')
    print(show(target, models=latent_models).to_string(index=False))
    print()

--- team_coordination ---
  CV                                         Model     R2  n     p     q
L2GO     K-Means clusters (no temporal, v20260905) -0.183 18 0.366 0.527
L2GO Continuous PCA (no discretization, v20260905) -0.411 18 0.396 0.561
L2GO                 HMM states new K5 (v20260905) -0.248 18 0.446 0.614
L2GO                     Baseline (task-mean only) -0.177 20 0.644 0.765
LOGO                 HMM states new K5 (v20260905) -0.173 18 0.238 0.328
LOGO Continuous PCA (no discretization, v20260905) -0.241 18 0.248 0.332
LOGO                     Baseline (task-mean only) -0.130 20 0.327 0.420
LOGO     K-Means clusters (no temporal, v20260905) -0.288 18 0.634 0.735

--- voice_inclusion ---
  CV                                         Model     R2  n     p     q
L2GO                     Baseline (task-mean only)  0.362 30 0.010 0.048
L2GO     K-Means clusters (no temporal, v20260905)  0.307 28 0.010 0.048
L2GO Continuous PCA (no discretization, v20260905)  0.332 28 0.020 0.058


**Interpretation:** none of the three latent/reduced representations reliably beats the task-mean baseline
on either headline target. For `voice_inclusion`, Continuous PCA edges past baseline under LOGO only
(R2=0.532 vs 0.357, p=0.02) but *not* under L2GO (0.332 vs 0.362) — inconsistent, not a replicated win.
HMM is consistently the **weakest** of the three across targets and CV schemes — it is the only one of
the three that goes strongly negative for `voice_inclusion` (L2GO R2=-0.199) and for `coord+coop`
(both CV schemes). For `mental_demand`, all three trail the baseline, reinforcing the earlier caveat that
`mental_demand`'s apparent signal is mostly a task effect. For `satisfaction`, K-Means comes closest to
baseline (0.30-0.31 vs 0.356/0.368) but still doesn't beat it.

**Bottom line:** adding *more* structure (HMM > K-Means > continuous PCA, in order of constraint) does
not help and, if anything, hurts — a clean, informative null across the *entire* latent-representation
family for RQ3, not just the specific HMM configuration. This should be reported as the main finding for
RQ3/H4, with the HMM-vs-Turn-taking comparison (H4 section above) as the sharper, single-model version
of the same conclusion.

### State stability check (H6)

A leave-groups-out refit check (5 disjoint folds, each holding out 2 of the 10 groups, full
scaler/PCA/HMM pipeline refit from scratch on the remaining 8 groups each time) tests whether the same
`k=5` states are reproducible rather than an artifact of fitting on all 10 groups at once. Agreement is
measured with the Adjusted Rand Index (ARI, chance-corrected, label-permutation-invariant) and percent
agreement after optimal state-label matching:

In [13]:
stability = pd.read_csv(REPO_ROOT / 'analysis' / 'results' / 'hmm_states_v20260905' / 'hmm_state_stability_v20260908.tsv', sep='\t')
print(stability.to_string(index=False))
print(f"\nmean ARI = {stability['ARI'].mean():.3f}  |  mean agreement (label-matched) = {stability['agreement_after_label_matching'].mean():.1%}")

 fold holdout_groups  n_train_groups  n_shared_windows      ARI  agreement_after_label_matching
    0  grp-07,grp-08               8               472 0.502007                        0.701271
    1  grp-09,grp-10               8               511 0.826757                        0.927593
    2  grp-11,grp-12               8               478 0.342927                        0.587866
    3  grp-13,grp-14               8               514 0.440792                        0.645914
    4  grp-15,grp-16               8               489 0.707955                        0.871166

mean ARI = 0.564  |  mean agreement (label-matched) = 74.7%


**Interpretation:** by conventional clustering-agreement benchmarks (ARI > 0.4-0.5 = strong, 0.2-0.4 =
moderate, < 0.2 = weak), mean ARI = 0.564 indicates the k=5 HMM states are **reasonably reproducible**
across different subsets of groups — not an artifact of fitting on all 10 groups at once, though
stability varies by fold (0.343-0.827). **This is a genuine positive result for H6**, distinct from and
not contradicted by the H4 predictive null: the states are a real, recoverable regularity in the data,
they simply do not carry additional predictive information for the self-reported outcomes beyond what
raw conversation-structure features already provide.

## Participant-level layer (exploratory, Sections 16b/16c/17) — reported for completeness

These use **participant-level rows** (broadcast group-task features onto each individual's own rating),
which sits outside the thesis's primary group-task-level design. Included here only as exploratory/
discussion material, with correct multiple-comparison handling:

- **16b — behavioral feature sets → participant outcomes:** 0/72 fits survive FDR correction. No robust
  participant-level behavioral signal.
- **16c — personality/demographic traits → `mental_demand`:** the only genuinely robust participant-level
  finding. Both trait sets survive even a conservative Bonferroni correction (8 tests total, alpha=0.00625):
  `BFI-44 + age` (LR_p=0.0003) and `BFI-44` alone (LR_p=0.0008), driven mainly by Openness (beta ~-0.86 to
  -1.02) and age (beta -0.033). This is a trait/covariate finding, not a group-interaction one — useful for
  a limitations/discussion paragraph, not for the core H2 claim.
- **17 — focal-vs-others individual-level design:** the two pre-registered confirmatory tests
  (`others_mean_eda_phasic_rate_hz`, `others_mean_tr_backchannel_count` → `voice_inclusion`) both came back
  null (p=0.91, p=0.78), as did two additional exploratory features tested afterward. Reported as a clean
  negative result: no evidence that *other* group members' behavior shifts a focal participant's own
  `voice_inclusion` rating in this dataset.

In [14]:
mlm16b = pd.read_csv(ANALYSIS_RES / 'participant_mixedlm_ablation_v20260907.tsv', sep='\t')
print(f'16b: {(mlm16b["LR_p"] < 0.05).sum()}/{len(mlm16b)} nominal p<0.05, '
      f'{(mlm16b["q"] < 0.05).sum()}/{len(mlm16b)} FDR-significant (q<0.05)')

print()
print('16c (reported from prior run — 8 tests total, Bonferroni alpha = 0.05/8 = 0.00625):')
trait_summary = pd.DataFrame([
    {'feature_set': 'BFI-44 + age', 'target': 'mental_demand', 'LR_p': 0.0003, 'survives_bonferroni': True},
    {'feature_set': 'BFI-44 (5 traits)', 'target': 'mental_demand', 'LR_p': 0.0008, 'survives_bonferroni': True},
])
print(trait_summary.to_string(index=False))

print()
confirm = pd.read_csv(ANALYSIS_RES / 'focal_others_confirmatory_pair_v20260908.tsv', sep='\t')
print('17 — pre-registered focal/others confirmatory pair + follow-up scan (all null):')
print(confirm[['hypothesis', 'target', 'n', 'coef', 'LR_p', 'q']].to_string(index=False))

16b: 5/72 nominal p<0.05, 0/72 FDR-significant (q<0.05)

16c (reported from prior run — 8 tests total, Bonferroni alpha = 0.05/8 = 0.00625):
      feature_set        target   LR_p  survives_bonferroni
     BFI-44 + age mental_demand 0.0003                 True
BFI-44 (5 traits) mental_demand 0.0008                 True

17 — pre-registered focal/others confirmatory pair + follow-up scan (all null):
                      hypothesis          target   n   coef   LR_p      q
  others_mean_eda_phasic_rate_hz voice_inclusion 109 -0.256 0.9133 0.9133
others_mean_tr_backchannel_count voice_inclusion 111  0.009 0.7839 0.9133
  others_mean_lex_agreement_rate voice_inclusion 111  2.012 0.7462 0.9133
   others_mean_tr_overlap_time_s voice_inclusion 111  0.012 0.4565 0.9133


## Recommended thesis narrative

1. **Headline result (H2):** group-level conversation-structure/participation features predict
   `team_coordination` (clean, no task confound) and `voice_inclusion` (beats task baseline) —
   this is the core, defensible "group interaction predicts group-functioning outcome" finding.
2. **Secondary/minor result:** the same feature family gives a small but FDR-significant edge on the
   composite `coord+coop` target.
3. **Explicit non-finding, reported honestly:** `mental_demand`/`satisfaction` are largely task-driven;
   claims of behavioral prediction there should be avoided or heavily qualified.
4. **Exploratory appendix:** participant-level focal/others analysis (Section 17) found no interpersonal
   effect on `voice_inclusion`; personality traits (not behavior) are the only robust participant-level
   predictor of `mental_demand` — useful as a limitations/future-work note, not a core claim.
5. **Limitations to state explicitly:** n=10 groups sharply limits model complexity and precision;
   associations are descriptive/predictive, not causal (per `docs/thesis_research_questions.md`).